# 16 · A glimpse of more: buoyant convection ☕🔥

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [⏳](/lite/notebooks/index.html?path=16-buoyant-convection.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/16-buoyant-convection.ipynb) |

<sub>✅ runs here · ⏳ runs but slow here · ❌ not available here</sub>


In notebooks 10 and 13 the coffee cooled passively. But a real cup *moves*: warm
fluid is lighter, so it **rises**, cool fluid **sinks**, and a circulation sets
in. We couple the **temperature** $T$ to a **flow** $\mathbf u$ through the
**Boussinesq** approximation — buoyancy enters the momentum balance as a force
proportional to temperature:
$$
-\Delta\mathbf u + \nabla p = \mathrm{Ra}\,T\,\hat{\mathbf y},\quad
\nabla\!\cdot\mathbf u = 0,\qquad
\partial_t T + \mathbf u\!\cdot\!\nabla T = \Delta T .
$$

The trick that keeps this **cheap and portable** (only the symmetric, positive
`sparsecholesky` solver — no indefinite or non-symmetric direct solver needed) is
the one the NGSolve i-tutorials use: march in **time** with an **IMEX** scheme —
treat the troublesome **convection explicitly** — and replace the Stokes
saddle-point by a **grad–div penalty**, so *every* solve is symmetric positive
definite.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw
import sys

def progress(i, n, label="working"):
    """A tiny dependency-free progress bar (survives JupyterLite / Colab / local)."""
    if (i + 1) % max(1, n // 100) == 0 or i + 1 == n:
        f = int(26 * (i + 1) / n)
        sys.stdout.write(f"\r  {label}… [{'█'*f}{'·'*(26-f)}] {100*(i+1)//n:3d}%")
        sys.stdout.flush()
        if i + 1 == n:
            sys.stdout.write("\n")

# a heated cavity: the cup cross-section, warm on one wall, cool on the other
Wb, Wt, H = 4.0, 5.0, 6.0
body = (WorkPlane().MoveTo(-Wb/2, 0).LineTo(Wb/2, 0)
        .LineTo(Wt/2, H).LineTo(-Wt/2, H).Close().Face())
body.edges.name = "insulated"
for e in body.edges:
    if   e.center[0] < -0.5: e.name = "warm"
    elif e.center[0] >  0.5: e.name = "cool"
mesh = Mesh(OCCGeometry(body, dim=2).GenerateMesh(maxh=0.4))
mesh.Curve(2)

## 1. Two fields, two symmetric operators

Velocity lives in a `VectorH1` (no-slip on every wall), temperature in an `H1`
(warm$=1$, cool$=0$). The **velocity** operator is a Stokes problem written with
a large **grad–div penalty** $\gamma$ instead of a pressure unknown — that makes
it symmetric positive definite and keeps $\nabla\!\cdot\mathbf u\approx0$. We
factorise it **once**.

In [ ]:
Ra, gamma = 1e4, 1e7
V = VectorH1(mesh, order=2, dirichlet="warm|cool|insulated")
S = H1(mesh, order=2, dirichlet="warm|cool")
u, vv = V.TnT()
T, St = S.TnT()

a_vel = BilinearForm(InnerProduct(Grad(u), Grad(vv))*dx
                     + gamma*div(u)*div(vv)*dx).Assemble()          # SPD
inv_vel = a_vel.mat.Inverse(V.FreeDofs(), inverse="sparsecholesky")

gfu = GridFunction(V)                                              # velocity
gfT = GridFunction(S)                                             # temperature
gfT.Set(0.5 - x/Wt)                                              # smooth start …
gfT.Set(mesh.BoundaryCF({"warm": 1.0, "cool": 0.0}), BND)        # … with warm/cool fixed

## 2. The IMEX time loop

Each step: (i) solve the **velocity** from the current temperature (buoyancy on
the right); (ii) advance the **temperature** by implicit Euler — diffusion
*implicit* (symmetric), the advection $\mathbf u\!\cdot\!\nabla T$ moved to the
right-hand side (**explicit**), plus a little **streamline diffusion** for
stability. Both solves are `sparsecholesky`. We store frames to animate.

In [ ]:
M  = BilinearForm(T*St*dx).Assemble()
dt, tau, nsteps = 0.008, 0.25, 400
gfT.AddMultiDimComponent(gfT.vec)                                 # frame 0
for step in range(nsteps):
    # (i) velocity from buoyancy  (symmetric solve)
    fbuo = LinearForm(Ra*gfT*vv[1]*dx).Assemble()
    gfu.vec.data = inv_vel * fbuo.vec

    # (ii) temperature: implicit diffusion (+streamline) — explicit convection
    Kdiff = BilinearForm(grad(T)*grad(St)*dx
                         + tau*(gfu*grad(T))*(gfu*grad(St))*dx).Assemble()
    mstar = M.mat.CreateMatrix()
    mstar.AsVector().data = M.mat.AsVector() + dt*Kdiff.mat.AsVector()
    inv_T = mstar.Inverse(S.FreeDofs(), inverse="sparsecholesky")
    conv = LinearForm((gfu*grad(gfT))*St*dx).Assemble()           # u·∇T, explicit
    res = (M.mat*gfT.vec - dt*conv.vec - mstar*gfT.vec).Evaluate()  # keeps Dirichlet
    gfT.vec.data += inv_T * res
    if step % 16 == 0:
        gfT.AddMultiDimComponent(gfT.vec)
    progress(step, nsteps, "convecting")

speed = Integrate(sqrt(gfu*gfu), mesh) / Integrate(CF(1), mesh)
print(f"reached a convecting state: mean flow speed {speed:.1f}, "
      f"∫(div u)² = {Integrate(div(gfu)**2, mesh):.0e}")
Draw(gfT, mesh, "temperature (press play)", interpolate_multidim=True, animate=True)

## 3. The convection roll

Warm fluid climbs the hot wall, drifts across the top, cools and sinks down the
cold wall — a single slow **convection roll**. The arrows make it plain.

In [ ]:
print("rises at the warm wall  (u_y > 0):", round(gfu(mesh(-1.5, 3))[1], 1))
print("sinks at the cool wall  (u_y < 0):", round(gfu(mesh( 1.5, 3))[1], 1))
Draw(gfu, mesh, "velocity — the convection roll", vectors={"grid_size": 26})

:::{dropdown} 🧠 Quiz — why treat convection *explicitly*, and what does the Rayleigh number do?
Folding the convection $\mathbf u\!\cdot\!\nabla T$ into the implicit matrix would
make it **non-symmetric**, demanding a direct *unsymmetric* solver (pardiso /
umfpack) that is not always available — and the Stokes pressure makes the
momentum block **indefinite** on top. Treating convection **explicitly** (IMEX)
and penalising $\nabla\!\cdot\mathbf u$ turns *every* solve symmetric positive
definite, so the cheap, ever-present `sparsecholesky` suffices — at the price of a
mild CFL limit on $\Delta t$. The **Rayleigh number** `Ra` sets the strength of
buoyancy against viscous + diffusive damping: below a critical value the fluid
sits still (pure conduction); above it convection switches on, and the higher
`Ra`, the faster and more intricate the flow — eventually unsteady and turbulent.
That is the doorway from a tutorial to real computational fluid dynamics.
:::

## Where the ☕ has taken us

From a cold Toblerone to a convecting cup, the same handful of ideas kept
returning — geometry, coefficient functions, spaces, weak forms, solvers — only
the physics grew richer. One last flourish remains before the expedition proper:
a nonlinear PDE on the **skin of the beast** itself — where the beast finally gets
its stripes.